In [1]:
import pandas as pd
import numpy as np 
import scanpy as sc 
import os
import scrublet as scr 
import glob

In [ ]:
# find . -type f | sort

In [2]:
study_files = {
    "Choudhury2022": {
        "matrix": "Data_Choudhury2022_Brain/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Choudhury2022_Brain/Genes.txt",
        "cells":  "Data_Choudhury2022_Brain/Cells.csv",
        "samples": "Data_Choudhury2022_Brain/Samples.csv",
        "units": "UMI"
    },
    "Couturier2020": {
        "matrix": "Data_Couturier2020_Brain/Exp_data_TPM.mtx",
        "genes":  "Data_Couturier2020_Brain/Genes.txt",
        "cells":  "Data_Couturier2020_Brain/Cells.csv",
        "samples": "Data_Couturier2020_Brain/Samples.csv",
        "units": "TPM"
    },
    "Darmanis2017": {
        "matrix": "Data_Darmanis2017_Brain/normalized_Exp_data_TPM.mtx",
        "genes":  "Data_Darmanis2017_Brain/Genes.txt",
        "cells":  "Data_Darmanis2017_Brain/Cells.csv",
        "samples": "Data_Darmanis2017_Brain/Samples.csv",
        "units": "TPM"          # normalized TPM is still TPM
    },
    "Filbin2018": {
        "matrix": "Data_Filbin2018_Brain/Exp_data_TPM.mtx",
        "genes":  "Data_Filbin2018_Brain/Genes.txt",
        "cells":  "Data_Filbin2018_Brain/Cells.csv",
        "samples": "Data_Filbin2018_Brain/Samples.csv",
        "units": "TPM"
    },
    "Gojo2020": {
        "matrix": "Data_Gojo2020_Brain/Exp_data_TPM.mtx",
        "genes":  "Data_Gojo2020_Brain/Genes.txt",
        "cells":  "Data_Gojo2020_Brain/Cells.csv",
        "samples": "Data_Gojo2020_Brain/Samples.csv",
        "units": "TPM"
    },
    "Hovestadt2019": {
        "matrix": "Data_Hovestadt2019_Brain/Exp_data_TPM.mtx",
        "genes":  "Data_Hovestadt2019_Brain/Genes.txt",
        "cells":  "Data_Hovestadt2019_Brain/Cells.csv",
        "samples": "Data_Hovestadt2019_Brain/Samples.csv",
        "units": "TPM"
    },
    "Neftel2019_10X": {
        "matrix": "Data_Neftel2019_Brain/10X/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Neftel2019_Brain/10X/Genes.txt",
        "cells":  "Data_Neftel2019_Brain/10X/Cells.csv",
        "samples": "Data_Neftel2019_Brain/Samples.csv",
        "units": "UMI"
    },
    "Neftel2019_SmartSeq2": {
        "matrix": "Data_Neftel2019_Brain/SmartSeq2/Exp_data_TPM.mtx",
        "genes":  "Data_Neftel2019_Brain/SmartSeq2/Genes.txt",
        "cells":  "Data_Neftel2019_Brain/SmartSeq2/Cells.csv",
        "samples": "Data_Neftel2019_Brain/Samples.csv",
        "units": "TPM"
    },
    "Tirosh2016": {
        "matrix": "Data_Tirosh2016_Brain/Exp_data_TPM.mtx",
        "genes":  "Data_Tirosh2016_Brain/Genes.txt",
        "cells":  "Data_Tirosh2016_Brain/Cells.csv",
        "samples": "Data_Tirosh2016_Brain/Samples.csv",
        "units": "TPM"
    },
    "Venteicher2017": {
        "matrix": "Data_Venteicher2017_Brain/Exp_data_TPM.mtx",
        "genes":  "Data_Venteicher2017_Brain/Genes.txt",
        "cells":  "Data_Venteicher2017_Brain/Cells.csv",
        "samples": "Data_Venteicher2017_Brain/Samples.csv",
        "units": "TPM"
    },
    "Wang2019": {
        "matrix": "Data_Wang2019_Brain/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Wang2019_Brain/Genes.txt",
        "cells":  "Data_Wang2019_Brain/Cells.csv",
        "samples": "Data_Wang2019_Brain/Samples.csv",
        "units": "UMI"
    },
    "Yuan2018": {
        "matrix": "Data_Yuan2018_Brain/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Yuan2018_Brain/Genes.txt",
        "cells":  "Data_Yuan2018_Brain/Cells.csv",
        "samples": "Data_Yuan2018_Brain/Samples.csv",
        "units": "UMI"
    }
}

In [ ]:
study_name = 'Zhang2022'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Ovarian'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)